# 01 Mdp Example

## 📚 Learning Objectives

By completing this notebook, you will:
- Formalize decision problems as MDPs
- Define states, actions, rewards, and transitions

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

## Official Structure Reference

This notebook belongs to **Course 09, Unit 1**.

---


## Lesson Brief

This lesson introduces the basic language of reinforcement learning through a very small Markov Decision Process (MDP).

You will see what a **state**, **action**, **reward**, **transition**, and **policy** actually mean before any algorithm appears.

Why this matters: if students do not understand the MDP structure first, later topics like value iteration and Q-learning become formulas without meaning.

This notebook is the foundation for the rest of Unit 1.

# Example 1: A Simple Markov Decision Process

This notebook gives you a concrete MDP before you move to policy evaluation,
policy iteration, and value iteration.

## What you should learn

By the end of this notebook, you should be able to:
- identify states, actions, rewards, and transitions in a simple problem
- describe an MDP using a small grid world
- read a policy as a mapping from states to actions
- understand why an MDP is the starting point for RL

## Before you start

You should already be comfortable with:
- basic Python syntax
- lists, dictionaries, and functions
- the RL terms `agent`, `environment`, and `reward`

## Study tip

Do not try to memorize definitions only.

Focus on answering this question:

`If I had to teach this problem to a computer, what information would I need to define first?`


## Inputs and Outputs

**Inputs**
- a tiny 3x3 grid world
- a small reward table
- a deterministic transition rule
- a hand-written example policy

**Outputs**
- a readable description of the MDP components
- example transitions from one state to the next
- a simple policy rollout showing how decisions accumulate reward


In [1]:
import numpy as np

GRID_SIZE = 3
ACTIONS = ["up", "right", "down", "left"]
ACTION_TO_DELTA = {
    "up": (-1, 0),
    "right": (0, 1),
    "down": (1, 0),
    "left": (0, -1),
}
START = (0, 0)
GOAL = (2, 2)
TRAP = (1, 1)
STEP_COST = -0.1

states = [(r, c) for r in range(GRID_SIZE) for c in range(GRID_SIZE)]
rewards = {state: STEP_COST for state in states}
rewards[GOAL] = 5.0
rewards[TRAP] = -3.0


def transition(state, action):
    if state in {GOAL, TRAP}:
        return state

    dr, dc = ACTION_TO_DELTA[action]
    nr = min(max(state[0] + dr, 0), GRID_SIZE - 1)
    nc = min(max(state[1] + dc, 0), GRID_SIZE - 1)
    return (nr, nc)


policy = {
    (0, 0): "right",
    (0, 1): "right",
    (0, 2): "down",
    (1, 0): "down",
    (1, 2): "down",
    (2, 0): "right",
    (2, 1): "right",
}

print("=" * 60)
print("Example 1: Markov Decision Process (MDP)")
print("=" * 60)

print("\n1. States")
print(states)

print("\n2. Actions")
print(ACTIONS)

print("\n3. Rewards")
for state in states:
    print(f"  {state}: {rewards[state]}")

print("\n4. Example transitions")
examples = [
    (START, "right"),
    ((0, 2), "down"),
    ((1, 0), "right"),
    ((2, 1), "right"),
]
for state, action in examples:
    next_state = transition(state, action)
    print(
        f"  state={state}, action={action:>5} -> next_state={next_state}, "
        f"reward={rewards[next_state]}"
    )

print("\n5. Example policy")
for state, action in policy.items():
    print(f"  {state} -> {action}")

print("\nCheckpoint questions:")
print("- Which parts of this problem belong to the environment?")
print("- Which part represents the agent's decision rule?")
print("- Why do we separate transitions from rewards?")



Example 1: Markov Decision Process (MDP)

1. States
[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]

2. Actions
['up', 'right', 'down', 'left']

3. Rewards
  (0, 0): -0.1
  (0, 1): -0.1
  (0, 2): -0.1
  (1, 0): -0.1
  (1, 1): -3.0
  (1, 2): -0.1
  (2, 0): -0.1
  (2, 1): -0.1
  (2, 2): 5.0

4. Example transitions
  state=(0, 0), action=right -> next_state=(0, 1), reward=-0.1
  state=(0, 2), action= down -> next_state=(1, 2), reward=-0.1
  state=(1, 0), action=right -> next_state=(1, 1), reward=-3.0
  state=(2, 1), action=right -> next_state=(2, 2), reward=5.0

5. Example policy
  (0, 0) -> right
  (0, 1) -> right
  (0, 2) -> down
  (1, 0) -> down
  (1, 2) -> down
  (2, 0) -> right
  (2, 1) -> right

Checkpoint questions:
- Which parts of this problem belong to the environment?
- Which part represents the agent's decision rule?
- Why do we separate transitions from rewards?


## Worked Example: Following a Policy Through the MDP

Now that the MDP is defined, the next step is to see what happens when an agent
actually follows a policy.

In the next cell, you will simulate one short episode starting from the start
state and observe how rewards accumulate over time.

In [2]:
def rollout(policy, start_state=START, max_steps=10):
    state = start_state
    trajectory = []
    total_reward = 0.0

    for step_idx in range(max_steps):
        if state in {GOAL, TRAP}:
            break

        action = policy[state]
        next_state = transition(state, action)
        reward = rewards[next_state]
        total_reward += reward
        trajectory.append((step_idx, state, action, next_state, reward))
        state = next_state

        if state in {GOAL, TRAP}:
            break

    return trajectory, total_reward, state


trajectory, total_reward, final_state = rollout(policy)

print("Policy rollout from the start state:\n")
for step_idx, state, action, next_state, reward in trajectory:
    print(
        f"step {step_idx}: state={state}, action={action:>5}, "
        f"next_state={next_state}, reward={reward}"
    )

print("\nFinal state:", final_state)
print("Total reward:", round(total_reward, 2))

print("\nReflection:")
print("- Did the policy avoid the trap?")
print("- What would happen if one action changed?")
print("- How could a better policy be discovered automatically?")

Policy rollout from the start state:

step 0: state=(0, 0), action=right, next_state=(0, 1), reward=-0.1
step 1: state=(0, 1), action=right, next_state=(0, 2), reward=-0.1
step 2: state=(0, 2), action= down, next_state=(1, 2), reward=-0.1
step 3: state=(1, 2), action= down, next_state=(2, 2), reward=5.0

Final state: (2, 2)
Total reward: 4.7

Reflection:
- Did the policy avoid the trap?
- What would happen if one action changed?
- How could a better policy be discovered automatically?


## Summary

You now have a concrete MDP in mind.

The important idea is not the grid itself. The important idea is the structure:
- states describe situations
- actions describe choices
- rewards describe feedback
- transitions describe how the world changes
- a policy describes how the agent behaves

### What comes next

In `02_mdp_solving.ipynb`, you will move from describing an MDP to solving one.

## 📚 References & Further Reading

**Books:**
- Sutton & Barto — [Reinforcement Learning: An Introduction](http://incompleteideas.net/book/the-book-2nd.html) (free online, the RL bible)

**Papers:**
- Mnih et al. (2015) — [Human-level control through deep RL (DQN)](https://www.nature.com/articles/nature14236)
- Silver et al. (2016) — [AlphaGo](https://www.nature.com/articles/nature16961)

**State-of-the-Art:** OpenAI Five beat world champions in Dota2; AlphaFold uses RL-like optimization for protein folding.

## Closing Takeaway

**Teaching takeaway:** An MDP is the language that turns a decision problem into something an RL algorithm can work with: states, actions, rewards, transitions, and a policy.

**If students remember one idea:** Before solving a problem, define the environment clearly. A weak MDP definition creates weak RL reasoning later.

**Quick check before moving on:**
- Can you point to the state, action, reward, and transition rule in this example without looking back at the definitions?
- Can you explain why a policy is different from the environment dynamics?

**Bridge to the next step:** Next, you will move from describing an MDP to actually solving one.
